# Setup Bronze tables

In [0]:
%reload_ext autoreload
%autoreload 2

import importlib
import etl_config.bronze_config as bronze_config_module

importlib.reload(bronze_config_module)

# Bronze table configuration

- Should be executed on setup
- Creating bronze table with base schema
- Compact existing small files
- Set table properties to enable auto compaction and optimize future writes
- Set columns description

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from utils.logger import get_logger
from etl_config.bronze_config import (
    CATALOG,
    BRONZE,
    BRONZE_CONFIG,
)

logger = get_logger("bronze_setup")

### Define base Bronze schemas

In [0]:
def create_base_schema(required_cols: list[str]) -> StructType:
    """Create schema with all required columns as STRING + audit columns."""
    fields = []
    
    # Required columns - all string
    for col in required_cols:
        fields.append(StructField(col, StringType(), True))
    
    # Audit columns
    fields.extend([
        StructField("_source_file", StringType(), True),
        StructField("_ingested_at", TimestampType(), True),
        StructField("_batch_id", StringType(), True),
    ])
    
    return StructType(fields)

### Create bronze tables

In [0]:
for table_key, cfg in BRONZE_CONFIG.items():
    table_name = cfg.table_name
    
    if not spark.catalog.tableExists(table_name):
        logger.info(f"Creating bronze table: {table_name}")
        
        # Create schema from required columns
        schema = create_base_schema(cfg.required_columns)
        
        # Create empty table
        df = spark.createDataFrame([], schema)
        df.write.format("delta").saveAsTable(table_name)
    

        # Set table properties to enable auto compaction and optimize future writes
        spark.sql(
            f"""
                ALTER TABLE {table_name}
                SET TBLPROPERTIES (
                    'delta.autoOptimize.optimizeWrite' = 'true',
                    'delta.autoOptimize.autoCompact' = 'true',
                    'delta.columnMapping.mode' = 'name',
                    'description' = :table_description
                )
            """,
            args={"table_description": cfg.table_description}
        )

        logger.info(f"Created: {table_name}")

    else:
        logger.info(f"Table already exists: {table_name}")
    
    # Add comments
    for column, comment in cfg.column_comments.items():
        try:
            spark.sql(
                f"COMMENT ON COLUMN {table_name}.{column} IS :comment",
                args={"comment": comment}
            )
            logger.info(f"Added comment to column: {table_name}.{column}")
        except:
            logger.info(f"Column still does not exists: {table_name}.{column}")

logger.info("Bronze table setup complete.")